# Ch 24. 디코딩 전략 — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 포함합니다.


## 문제 1

**문제**: Greedy, Top-k=5, Top-p=0.9, Temperature=0.7로 같은 프롬프트에서 생성 비교하라.

### 풀이

Greedy는 최댓값을 결정적으로 택한다. 나머지는 정규화된 후보 집합에서 표본화한다. 공정한 비교는 같은 로짓과 시드를 사용해 고유 토큰 수와 로그확률을 함께 보고한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
import numpy as np
rng=np.random.default_rng(24); z=np.array([3.,2.2,1.5,1.,.4,-.2]);
def sample(z, mode):
 p=np.exp(z-z.max()); p/=p.sum()
 if mode=='greedy': return int(p.argmax())
 if mode=='topk': idx=np.argsort(p)[-5:]
 else:
  idx=np.argsort(p)[::-1]; idx=idx[np.cumsum(p[idx])-p[idx] < .9]
 q=p[idx]/p[idx].sum(); return int(rng.choice(idx,p=q))
out={m:[sample(z/.7 if m=='temp' else z, 'topp' if m=='temp' else m) for _ in range(200)] for m in ['greedy','topk','topp','temp']}
assert len(set(out['greedy']))==1 and all(0<=x<6 for v in out.values() for x in v)
{k:len(set(v)) for k,v in out.items()}


## 문제 2

**문제**: Temperature 0.3, 0.7, 1.0, 1.5에 따른 엔트로피와 생성 다양성을 비교하라.

### 풀이

온도 $T$는 $p_i(T)=\mathrm{softmax}(z_i/T)$를 만든다. $T$가 작으면 분포가 집중되어 엔트로피와 다양성이 대체로 감소하고, 크면 평탄해져 증가한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
import numpy as np
z=np.array([3.,2.,1.,0.,-1.]); rows=[]
for T in [.3,.7,1.,1.5]:
 p=np.exp(z/T-(z/T).max()); p/=p.sum(); H=-np.sum(p*np.log(p)); rows.append((T,H,1/np.sum(p*p)))
assert all(rows[i][1] < rows[i+1][1] for i in range(3)); rows


## 문제 3

**문제**: Beam Search의 빔 폭 1, 4, 8을 비교하라.

### 풀이

빔 탐색은 폭 $B$마다 누적 로그확률 상위 $B$개 접두사를 유지한다. $B=1$은 greedy이고, $B$가 커지면 탐색 점수는 나빠지지 않지만 길이 편향과 다양성 감소가 생길 수 있다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
trans={():[(0,-.1),(1,-.2)],(0,):[(0,-2),(1,-.1)],(1,):[(0,-.2),(1,-.4)]}
def beam(B):
 beams=[((),0.)]
 for _ in range(2):
  cand=[]
  for seq,s in beams:
   opts=trans.get(seq,[(0,-.3),(1,-.3)])
   cand += [(seq+(t,),s+lp) for t,lp in opts]
  beams=sorted(cand,key=lambda x:x[1],reverse=True)[:B]
 return beams[0]
scores=[beam(B)[1] for B in [1,4,8]]
assert scores[0] <= scores[1] <= scores[2]; scores


## 문제 4

**문제**: Repetition Penalty 1.0, 1.1, 1.3의 효과를 실험하라.

### 풀이

각 조건에 동일한 seed별 uniform 난수열을 재사용하는 common-random-numbers 비교를 수행한다. 64개 seed의 반복률 평균과 표준편차, 그리고 penalty 1.0 대비 paired 차이의 평균·표준오차를 보고한다. 표본 평균의 완전한 단조성은 강제하지 않고, 독립적으로 계산한 첫 반복 토큰 확률이 penalty 증가에 따라 감소하는지와 가장 강한 penalty의 paired 평균 효과를 검증한다.

아래 코드는 외부 다운로드가 없는 소규모 CPU 실험이며, 실행 결과에서 실제 통계량을 출력한다.


In [ ]:
import numpy as np

base = np.array([2.5, 2.0, 1.0, 0.0])
penalties = (1.0, 1.1, 1.3)

def repeat_rate(penalty, uniforms, steps=400):
    seq = []
    for u in uniforms[:steps]:
        logits = base.copy()
        if seq:
            repeated = np.unique(seq)
            positive = logits[repeated] > 0
            logits[repeated[positive]] /= penalty
            logits[repeated[~positive]] *= penalty
        probs = np.exp(logits - logits.max())
        probs /= probs.sum()
        seq.append(int(np.searchsorted(np.cumsum(probs), u, side='right')))
    return np.mean(np.asarray(seq[1:]) == np.asarray(seq[:-1]))

seed_rates = {p: [] for p in penalties}
for seed in range(64):
    matched_uniforms = np.random.default_rng(seed).random(400)
    for penalty in penalties:
        seed_rates[penalty].append(repeat_rate(penalty, matched_uniforms))

summary = {
    p: {'mean': float(np.mean(v)), 'std': float(np.std(v, ddof=1))}
    for p, v in seed_rates.items()
}
paired = np.asarray(seed_rates[1.3]) - np.asarray(seed_rates[1.0])
summary['paired_1.3_minus_1.0'] = {
    'mean': float(paired.mean()),
    'standard_error': float(paired.std(ddof=1) / np.sqrt(len(paired))),
}

def probability_of_immediate_repeat(penalty, previous=0):
    logits = base.copy()
    logits[previous] /= penalty
    probs = np.exp(logits - logits.max())
    return float(probs[previous] / probs.sum())

reference = [probability_of_immediate_repeat(p) for p in penalties]
assert reference[0] > reference[1] > reference[2]
assert summary['paired_1.3_minus_1.0']['mean'] < 0
summary


## 문제 5

**문제**: Speculative Decoding이 2-3x 빠른 이유를 수학적으로 설명하라.

### 풀이

초안 모델이 한 번에 $k$개를 제안하고 목표 모델이 병렬 검증한다. 평균 승인 수를 $a$라 하면 토큰당 목표 모델 호출 비용은 대략 $1/(1+ak)$이고, 초안 비용 $c_d$를 포함한 속도 향상은 $S\approx (1+ak)/(1+k c_d)$이다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
def speedup(k,accept,draft_cost): return (1+k*accept)/(1+k*draft_cost)
vals=[speedup(4,a,.05) for a in [.5,.7,.9]]
assert vals[0] > 2 and vals[-1] < 4; vals
